# 2. Hyperspectral retrieval: spectral projections

Run the cells in order. This demonstration uses synthetic intensities; iteration counts are illustrative, not convergence guarantees. Replace the data-generation cell with your normalized experimental intensities, support and detector mask. See [the examples guide](../docs/universal_reconstruction_examples.md).

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "library" / "phase_retrieval_universal.py").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from library import phase_retrieval_universal as universal

# Small synthetic data make this notebook runnable without experimental files.
shape = (32, 32)
y, x = np.indices(shape)
supportmask = ((x - 16)**2 + (y - 16)**2 < 7**2).astype(float)
mask_pixel = np.zeros(shape, dtype=int)  # 0 measured; 1 invalid/unmeasured.
magnetic_pattern = np.tanh((x - 16) / 2) * supportmask

def to_field(obj):
    return np.fft.ifftshift(np.fft.ifft2(np.fft.fftshift(obj)))

def to_object(field):
    return np.fft.ifftshift(np.fft.fft2(np.fft.fftshift(field)))


In [ ]:
energies_eV = np.array([770., 780., 790.])
response = np.array([-0.1 + 0.1j, -0.3 + 0.2j, -0.15 + 0.3j])
objects_true = np.stack([supportmask * np.exp(q * (1 + 0.3 * magnetic_pattern)) for q in response])
holograms = np.stack([abs(to_field(obj))**2 for obj in objects_true])
n = len(energies_eV)


Choose `none` for independent retrieval, `svd` for low-rank log-objects, or `rank1_spectral` for a shared spatial component and spectrum. SVD/rank-one require a pure energy scan: fixed state, polarization and illumination, with one observation per distinct energy. Use notebook 03 for mixed metadata.

In [ ]:
recipe = universal.default_universal_phase_retrieval_recipe()
recipe.update(
    modes=[1], mode_initialization="support_fft",
    warmup_mode=["HAPRE", "ER"], warmup_Nit=[40, 10],
    inner_mode=["ER"], inner_Nit=[10], outer_iterations=3,
    shuffle_observations=False,
    projection_relaxation=0.3,
    final_projection_relaxation=0,
    random_seed=0,
)
recipe.update(projection_model="svd", rank=1,
              energy_values=energies_eV, projection_every=n)
# Alternative, before running:
# recipe.update(projection_model="rank1_spectral", spectral_constraint="free")
# For suitable spectral coverage: spectral_constraint="kk".
fields, warmup, components, bsmasks, errors = universal.universal_phase_retrieval_algorithm(
    holograms, mask_pixel, supportmask,
    state_labels=["sample"] * n, energy_labels=energies_eV,
    polarization_coefficients=[+1] * n, illumination_labels=["beam"] * n,
    universal_recipe=recipe,
)
fig, axes = plt.subplots(1, n, figsize=(9, 3))
for i, ax in enumerate(axes):
    ax.imshow(abs(to_object(fields[i])), cmap="magma")
    ax.set_title(f"{energies_eV[i]:g} eV")
plt.tight_layout(); plt.show()
